
# GVH Diagonal Cubic 0.2.13 — Hamiltonian Constraints and Degrees of Freedom
## Analyse de Dirac–Bergmann, contraintes et comptage des degrés de liberté

**Auteur : Charlemagne O Laurince**

---

## Objectif

Le notebook 0.2.12 a introduit un modèle réduit pour les cinq composantes spatiales, symétriques et sans trace du champ directionnel :

\[
q_A(t,\mathbf x),
\qquad
A=1,\dots,5.
\]

La dynamique libre minimale était :

\[
\mathcal L_D^{(2)}
=
\frac{K_D}{2}\dot q_A\dot q_A
-
\frac{G_D}{2}\partial_iq_A\partial_iq_A
-
\frac{m_D^2}{2}q_Aq_A.
\]

Cette version étudie la structure hamiltonienne :

\[
L
\longrightarrow
p_A
\longrightarrow
H_C
\longrightarrow
\phi_a
\longrightarrow
\{\phi_a,\phi_b\}
\longrightarrow
N_{\mathrm{DOF}}.
\]

---

## Limite scientifique

Le calcul complet d'un tenseur covariant \(D_{\mu\nu}\) couplé à la métrique exige une décomposition \(3+1\) de l'action covariante définitive, qui n'est pas encore disponible.

Le présent notebook fournit :

1. l'analyse exacte du modèle réduit à cinq modes ;
2. des prototypes contraints de Dirac–Bergmann ;
3. les critères que devra satisfaire l'analyse covariante complète.

Il ne prétend donc pas encore démontrer le nombre définitif de degrés de liberté de toute théorie GVH.


In [1]:

import sympy as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sp.init_printing()
np.set_printoptions(precision=12, suppress=True)

print("Environnement chargé.")


Environnement chargé.



# 1. Rappel : cinq modes sans trace

Dans une base locale spatiale :

\[
\bar D_{ij}
=
\begin{pmatrix}
q_1+q_2/\sqrt3 & q_3 & q_4\\
q_3 & -q_1+q_2/\sqrt3 & q_5\\
q_4 & q_5 & -2q_2/\sqrt3
\end{pmatrix}.
\]

Cette paramétrisation impose automatiquement :

\[
\bar D_{ij}=\bar D_{ji},
\qquad
\delta^{ij}\bar D_{ij}=0.
\]


In [2]:

q1, q2, q3, q4, q5 = sp.symbols("q1 q2 q3 q4 q5", real=True)
sqrt3 = sp.sqrt(3)

Dbar = sp.Matrix([
    [q1 + q2/sqrt3, q3, q4],
    [q3, -q1 + q2/sqrt3, q5],
    [q4, q5, -2*q2/sqrt3]
])

I2 = sp.simplify(sp.trace(Dbar*Dbar))

print("Trace :")
display(sp.simplify(sp.trace(Dbar)))

print("Invariant quadratique :")
display(sp.factor(I2))


Trace :


0

Invariant quadratique :


  ⎛  2     2     2     2     2⎞
2⋅⎝q₁  + q₂  + q₃  + q₄  + q₅ ⎠


# 2. Modèle homogène libre

Pour le comptage hamiltonien initial, nous supprimons les gradients spatiaux :

\[
L_{\mathrm{hom}}
=
\sum_{A=1}^{5}
\left[
\frac{K_D}{2}\dot q_A^2
-
\frac{m_D^2}{2}q_A^2
\right].
\]

Les moments conjugués sont :

\[
p_A
=
\frac{\partial L}{\partial\dot q_A}
=
K_D\dot q_A.
\]

Lorsque \(K_D\neq0\), la transformation de Legendre est inversible.


In [3]:

t = sp.symbols("t", real=True)
K_D, m_D = sp.symbols("K_D m_D", nonzero=True, real=True)

q = [sp.Function(f"q{i}")(t) for i in range(1, 6)]
qdot = [sp.diff(qi, t) for qi in q]

L_hom = sum(
    sp.Rational(1,2)*K_D*vi**2
    - sp.Rational(1,2)*m_D**2*qi**2
    for qi, vi in zip(q, qdot)
)

momenta = [
    sp.simplify(sp.diff(L_hom, vi))
    for vi in qdot
]

for i, pi in enumerate(momenta, start=1):
    print(f"p_{i} =")
    display(pi)


p_1 =


    d        
K_D⋅──(q₁(t))
    dt       

p_2 =


    d        
K_D⋅──(q₂(t))
    dt       

p_3 =


    d        
K_D⋅──(q₃(t))
    dt       

p_4 =


    d        
K_D⋅──(q₄(t))
    dt       

p_5 =


    d        
K_D⋅──(q₅(t))
    dt       


# 3. Hessienne et régularité de Legendre

La hessienne cinétique est :

\[
W_{AB}
=
\frac{\partial^2L}
{\partial\dot q_A\partial\dot q_B}
=
K_D\delta_{AB}.
\]

Son déterminant est :

\[
\det W=K_D^5.
\]

Ainsi :

- \(K_D\neq0\) : système régulier, aucune contrainte primaire issue des cinq modes ;
- \(K_D=0\) : système singulier, apparition de contraintes primaires.


In [4]:

W = sp.Matrix([
    [
        sp.diff(sp.diff(L_hom, vi), vj)
        for vj in qdot
    ]
    for vi in qdot
])

det_W = sp.factor(W.det())
rank_W_generic = W.rank()

print("Hessienne :")
display(W)

print("Déterminant :")
display(det_W)

print("Rang générique :", rank_W_generic)


Hessienne :


⎡K_D   0    0    0    0 ⎤
⎢                       ⎥
⎢ 0   K_D   0    0    0 ⎥
⎢                       ⎥
⎢ 0    0   K_D   0    0 ⎥
⎢                       ⎥
⎢ 0    0    0   K_D   0 ⎥
⎢                       ⎥
⎣ 0    0    0    0   K_D⎦

Déterminant :


   5
K_D 

Rang générique : 5



# 4. Hamiltonien canonique régulier

Pour \(K_D\neq0\) :

\[
\dot q_A=\frac{p_A}{K_D}.
\]

L'hamiltonien canonique est :

\[
H_C
=
\sum_Ap_A\dot q_A-L
=
\sum_A
\left[
\frac{p_A^2}{2K_D}
+
\frac{m_D^2}{2}q_A^2
\right].
\]

La positivité de l'énergie exige au niveau libre :

\[
K_D>0,
\qquad
m_D^2\ge0.
\]


In [5]:

p_symbols = sp.symbols("p1:6", real=True)

velocity_solution = {
    qdot[i]: p_symbols[i]/K_D
    for i in range(5)
}

H_canonical = sp.simplify(
    sum(p_symbols[i]*qdot[i] for i in range(5))
    - L_hom
)

H_canonical = sp.simplify(
    H_canonical.subs(velocity_solution)
)

print("Hamiltonien canonique :")
display(H_canonical)


Hamiltonien canonique :


       2 ⎛  2        2        2        2        2   ⎞     2     2     2     2  ↪
K_D⋅m_D ⋅⎝q₁ (t) + q₂ (t) + q₃ (t) + q₄ (t) + q₅ (t)⎠ + p₁  + p₂  + p₃  + p₄   ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                       2⋅K_D                                   ↪

↪     2
↪ + p₅ 
↪ ─────
↪      


# 5. Crochet de Poisson

Pour des variables canoniques :

\[
\{q_A,p_B\}=\delta_{AB},
\qquad
\{q_A,q_B\}=0,
\qquad
\{p_A,p_B\}=0.
\]

Le crochet général est :

\[
\{F,G\}
=
\sum_A
\left(
\frac{\partial F}{\partial q_A}
\frac{\partial G}{\partial p_A}
-
\frac{\partial F}{\partial p_A}
\frac{\partial G}{\partial q_A}
\right).
\]


In [6]:

q_symbols = sp.symbols("Q1:6", real=True)

def poisson_bracket(F, G, qs, ps):
    result = sp.Integer(0)
    for qi, pi in zip(qs, ps):
        result += (
            sp.diff(F, qi)*sp.diff(G, pi)
            - sp.diff(F, pi)*sp.diff(G, qi)
        )
    return sp.simplify(result)

canonical_tests = []
for i in range(5):
    for j in range(5):
        canonical_tests.append({
            "i": i+1,
            "j": j+1,
            "{Qi,Pj}": poisson_bracket(
                q_symbols[i],
                p_symbols[j],
                q_symbols,
                p_symbols
            )
        })

pd.DataFrame(canonical_tests).head(10)


,i,j,"{Qi,Pj}"
0,1,1,1
1,1,2,0
2,1,3,0
3,1,4,0
4,1,5,0
5,2,1,0
6,2,2,1
7,2,3,0
8,2,4,0
9,2,5,0



# 6. Équations de Hamilton

\[
\dot q_A=\{q_A,H_C\},
\]

\[
\dot p_A=\{p_A,H_C\}.
\]

Elles doivent reproduire :

\[
\dot q_A=\frac{p_A}{K_D},
\qquad
\dot p_A=-m_D^2q_A.
\]

Donc :

\[
K_D\ddot q_A+m_D^2q_A=0.
\]


In [7]:

H_phase = sum(
    p_symbols[i]**2/(2*K_D)
    + m_D**2*q_symbols[i]**2/2
    for i in range(5)
)

hamilton_rows = []

for i in range(5):
    qdot_H = poisson_bracket(
        q_symbols[i], H_phase,
        q_symbols, p_symbols
    )
    pdot_H = poisson_bracket(
        p_symbols[i], H_phase,
        q_symbols, p_symbols
    )
    hamilton_rows.append({
        "mode": i+1,
        "qdot": qdot_H,
        "pdot": pdot_H
    })

pd.DataFrame(hamilton_rows)


,mode,qdot,pdot
0,1,p1/K_D,-Q1*m_D**2
1,2,p2/K_D,-Q2*m_D**2
2,3,p3/K_D,-Q3*m_D**2
3,4,p4/K_D,-Q4*m_D**2
4,5,p5/K_D,-Q5*m_D**2



# 7. Formule générale de comptage des degrés de liberté

Pour un espace des phases de dimension \(2N\), avec :

- \(F\) contraintes de première classe ;
- \(S\) contraintes de deuxième classe ;

le nombre de degrés de liberté physiques est :

\[
N_{\mathrm{DOF}}
=
N-F-\frac{S}{2}.
\]

Équivalent :

\[
N_{\mathrm{DOF}}
=
\frac{2N-2F-S}{2}.
\]

Pour le modèle régulier libre :

\[
N=5,
\qquad
F=0,
\qquad
S=0,
\]

donc :

\[
N_{\mathrm{DOF}}=5.
\]


In [8]:

def count_dof(N_config, first_class, second_class):
    return N_config - first_class - second_class/2

dof_regular = count_dof(
    N_config=5,
    first_class=0,
    second_class=0
)

print("DOF du modèle libre régulier :", dof_regular)


DOF du modèle libre régulier : 5.0



# 8. Prototype avec multiplicateur de Lagrange

Pour illustrer l'algorithme de Dirac–Bergmann, ajoutons un multiplicateur \(\lambda(t)\) imposant :

\[
C(q)=q_1+q_2=0.
\]

Le lagrangien devient :

\[
L_{\lambda}
=
L_{\mathrm{hom}}
+
\lambda(q_1+q_2).
\]

Comme \(\dot\lambda\) n'apparaît pas :

\[
p_\lambda=0
\]

est une contrainte primaire.


In [9]:

lam = sp.Function("lam")(t)
lamdot = sp.diff(lam, t)

L_constrained = L_hom + lam*(q[0] + q[1])

p_lambda = sp.simplify(
    sp.diff(L_constrained, lamdot)
)

print("Contrainte primaire p_lambda =")
display(p_lambda)


Contrainte primaire p_lambda =


0


# 9. Hamiltonien total et contrainte secondaire

L'hamiltonien total est :

\[
H_T
=
H_C
-
\lambda(q_1+q_2)
+
u_\lambda p_\lambda.
\]

La préservation temporelle de la contrainte primaire impose :

\[
\dot p_\lambda
=
\{p_\lambda,H_T\}
\approx0,
\]

ce qui produit la contrainte secondaire :

\[
C_1=q_1+q_2\approx0.
\]

Sa préservation donne ensuite :

\[
C_2=p_1+p_2\approx0.
\]


In [10]:

Qlam, Plam, Ulam, Lam = sp.symbols(
    "Qlam Plam Ulam Lam",
    real=True
)

qs_extended = list(q_symbols) + [Qlam]
ps_extended = list(p_symbols) + [Plam]

C_primary = Plam
C1 = q_symbols[0] + q_symbols[1]

H_total = (
    H_phase
    - Lam*C1
    + Ulam*C_primary
)

secondary_from_primary = poisson_bracket(
    C_primary,
    H_total,
    qs_extended,
    ps_extended
)

# Dans cette représentation, Lam joue le rôle de coordonnée Qlam.
H_total_coordinate = (
    H_phase
    - Qlam*C1
    + Ulam*C_primary
)

secondary_from_primary = poisson_bracket(
    C_primary,
    H_total_coordinate,
    qs_extended,
    ps_extended
)

C2 = sp.simplify(
    poisson_bracket(
        C1,
        H_total_coordinate,
        qs_extended,
        ps_extended
    )
)

print("Préservation de p_lambda :")
display(secondary_from_primary)

print("Contrainte suivante C2 :")
display(C2)


Préservation de p_lambda :


Q₁ + Q₂

Contrainte suivante C2 :


p₁ + p₂
───────
  K_D  


# 10. Matrice des crochets de contraintes

Considérons :

\[
\phi_1=p_\lambda,
\qquad
\phi_2=q_1+q_2,
\qquad
\phi_3=p_1+p_2.
\]

La matrice de Dirac est :

\[
\Delta_{ab}
=
\{\phi_a,\phi_b\}.
\]

Son rang permet d'identifier les contraintes de deuxième classe.


In [11]:

constraints = [
    C_primary,
    C1,
    C2
]

Delta = sp.Matrix([
    [
        poisson_bracket(
            constraints[a],
            constraints[b],
            qs_extended,
            ps_extended
        )
        for b in range(len(constraints))
    ]
    for a in range(len(constraints))
])

print("Matrice de contraintes :")
display(Delta)

print("Rang :")
display(Delta.rank())


Matrice de contraintes :


⎡0   0    0 ⎤
⎢           ⎥
⎢         2 ⎥
⎢0   0   ───⎥
⎢        K_D⎥
⎢           ⎥
⎢   -2      ⎥
⎢0  ───   0 ⎥
⎣   K_D     ⎦

Rang :


2


# 11. Interprétation du prototype contraint

Dans ce prototype :

- \(p_\lambda\) est associé au multiplicateur ;
- \(C_1=q_1+q_2\) et \(C_2=p_1+p_2\) forment une paire de deuxième classe car :

\[
\{C_1,C_2\}=2.
\]

La paire retire un degré de liberté de configuration.

Le multiplicateur \(\lambda\) et son moment ne représentent pas un mode physique.

Le secteur physique des cinq \(q_A\) passe donc de cinq à quatre modes dans cet exemple.


In [12]:

dof_constrained_physical_sector = count_dof(
    N_config=5,
    first_class=0,
    second_class=2
)

print(
    "DOF du secteur q_A après la paire de deuxième classe :",
    dof_constrained_physical_sector
)


DOF du secteur q_A après la paire de deuxième classe : 4.0



# 12. Prototype de première classe

Une contrainte de première classe génère une redondance de jauge.

Considérons un système jouet avec deux coordonnées \((x,y)\) et la contrainte :

\[
\phi=p_y\approx0,
\]

si l'hamiltonien ne dépend ni de \(y\) ni de \(p_y\).

Alors :

\[
\{\phi,H\}=0,
\]

et \(\phi\) est de première classe.

Pour :

\[
N=2,
\qquad
F=1,
\qquad
S=0,
\]

on obtient :

\[
N_{\mathrm{DOF}}=1.
\]


In [13]:

x, y, px, py = sp.symbols(
    "x y px py",
    real=True
)

H_gauge = px**2/2 + x**2/2
phi_gauge = py

gauge_preservation = poisson_bracket(
    phi_gauge,
    H_gauge,
    [x, y],
    [px, py]
)

dof_gauge = count_dof(
    N_config=2,
    first_class=1,
    second_class=0
)

print("{p_y,H} =", gauge_preservation)
print("DOF du système jouet :", dof_gauge)


{p_y,H} = 0
DOF du système jouet : 1.0



# 13. Cas singulier \(K_D=0\)

Si le coefficient cinétique s'annule :

\[
K_D=0,
\]

alors :

\[
p_A=0
\]

pour les cinq modes.

Cela produit cinq contraintes primaires.

La préservation temporelle peut imposer :

\[
m_D^2q_A=0.
\]

Si \(m_D^2\neq0\), les contraintes peuvent éliminer tous les modes.

Ce cas n'est donc pas une limite dynamique régulière du modèle libre.


In [14]:

singular_case = pd.DataFrame([
    {
        "condition": "K_D != 0",
        "primary_constraints": 0,
        "interpretation": "regular five-mode sector"
    },
    {
        "condition": "K_D = 0, m_D^2 != 0",
        "primary_constraints": 5,
        "interpretation": "algebraic/non-propagating sector"
    },
    {
        "condition": "K_D = 0, m_D^2 = 0",
        "primary_constraints": 5,
        "interpretation": "highly degenerate; gauge/strong-coupling analysis required"
    }
])

singular_case


,condition,primary_constraints,interpretation
0,K_D != 0,0,regular five-mode sector
1,"K_D = 0, m_D^2 != 0",5,algebraic/non-propagating sector
2,"K_D = 0, m_D^2 = 0",5,highly degenerate; gauge/strong-coupling analy...



# 14. Positivité hamiltonienne et ghost

Pour le secteur libre :

\[
H_D
=
\sum_A
\left[
\frac{p_A^2}{2K_D}
+
\frac{m_D^2}{2}q_A^2
\right].
\]

- \(K_D<0\) rend le terme cinétique hamiltonien non borné inférieurement ;
- \(m_D^2<0\) rend l'origine instable, mais ne constitue pas nécessairement un ghost ;
- une analyse complète doit inclure les couplages et les contraintes.

Le diagnostic ghost/tachyon doit donc rester séparé.


In [15]:

def energy_classification(K, m2):
    if K < 0:
        return "GHOST-RISK"
    if K == 0:
        return "DEGENERATE"
    if m2 < 0:
        return "TACHYONIC-INSTABILITY"
    return "POSITIVE-FREE-SECTOR"

energy_scan = pd.DataFrame([
    {
        "K_D": K,
        "m_D2": m2,
        "classification": energy_classification(K, m2)
    }
    for K in [-1.0, 0.0, 0.5, 1.0]
    for m2 in [-1.0, 0.0, 1.0]
])

energy_scan


,K_D,m_D2,classification
0,-1.0,-1.0,GHOST-RISK
1,-1.0,0.0,GHOST-RISK
2,-1.0,1.0,GHOST-RISK
3,0.0,-1.0,DEGENERATE
4,0.0,0.0,DEGENERATE
5,0.0,1.0,DEGENERATE
6,0.5,-1.0,TACHYONIC-INSTABILITY
7,0.5,0.0,POSITIVE-FREE-SECTOR
8,0.5,1.0,POSITIVE-FREE-SECTOR
9,1.0,-1.0,TACHYONIC-INSTABILITY



# 15. Extension avec gradients spatiaux

Pour un mode de Fourier \(\mathbf k\) :

\[
H_{\mathbf k}
=
\sum_A
\left[
\frac{|p_A|^2}{2K_D}
+
\frac12
\left(
G_Dk^2+m_D^2
\right)
|q_A|^2
\right].
\]

La stabilité du secteur libre exige :

\[
K_D>0,
\]

\[
G_D\ge0,
\]

\[
m_D^2\ge0
\]

pour tous les nombres d'onde.

La vitesse effective est :

\[
c_D^2=\frac{G_D}{K_D}.
\]


In [16]:

G_D, k = sp.symbols(
    "G_D k",
    real=True
)

omega2 = sp.simplify(
    (G_D*k**2 + m_D**2)/K_D
)

print("Relation de dispersion :")
display(omega2)


Relation de dispersion :


     2      2
G_D⋅k  + m_D 
─────────────
     K_D     


# 16. Couplage à une variable métrique réduite

Pour préparer le futur couplage gravitationnel, considérons une variable métrique réduite \(a(t)\) et un mode directionnel \(q(t)\) :

\[
L
=
-\frac{3M^2a\dot a^2}{N}
+
\frac{a^3K_D}{2N}\dot q^2
-
Na^3V(q).
\]

Le lapse \(N(t)\) ne possède pas de vitesse.

Ainsi :

\[
p_N=0
\]

est une contrainte primaire, et sa préservation produit une contrainte hamiltonienne.


In [17]:

a = sp.Function("a")(t)
N = sp.Function("N")(t)
qred = sp.Function("qred")(t)

Mpl, KD_red = sp.symbols(
    "Mpl KD_red",
    positive=True
)

Vred = sp.Function("Vred")

L_mini = (
    -3*Mpl**2*a*sp.diff(a,t)**2/N
    + a**3*KD_red*sp.diff(qred,t)**2/(2*N)
    - N*a**3*Vred(qred)
)

p_a = sp.simplify(
    sp.diff(L_mini, sp.diff(a,t))
)

p_q = sp.simplify(
    sp.diff(L_mini, sp.diff(qred,t))
)

p_N = sp.simplify(
    sp.diff(L_mini, sp.diff(N,t))
)

constraint_N = sp.simplify(
    sp.diff(L_mini, N)
)

print("p_a =")
display(p_a)

print("p_q =")
display(p_q)

print("p_N =")
display(p_N)

print("Contrainte issue de N :")
display(sp.factor(constraint_N))


p_a =


      2      d        
-6⋅Mpl ⋅a(t)⋅──(a(t)) 
             dt       
──────────────────────
         N(t)         

p_q =


        3    d          
KD_red⋅a (t)⋅──(qred(t))
             dt         
────────────────────────
          N(t)          

p_N =


0

Contrainte issue de N :


⎛                            2                    2                            ↪
⎜          2    ⎛d          ⎞         2 ⎛d       ⎞       2                   2 ↪
⎜- KD_red⋅a (t)⋅⎜──(qred(t))⎟  + 6⋅Mpl ⋅⎜──(a(t))⎟  - 2⋅N (t)⋅Vred(qred(t))⋅a  ↪
⎝               ⎝dt         ⎠           ⎝dt      ⎠                             ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                           2                                   ↪
                                        2⋅N (t)                                ↪

↪    ⎞     
↪    ⎟     
↪ (t)⎟⋅a(t)
↪    ⎠     
↪ ─────────
↪          
↪          


# 17. Ce que montre le modèle réduit métrique–directionnel

Le modèle réduit confirme une structure classique :

1. le lapse \(N\) produit une contrainte primaire ;
2. sa préservation impose la contrainte hamiltonienne ;
3. l'invariance par reparamétrisation temporelle réduit le nombre de variables physiques ;
4. le comptage naïf des composantes ne suffit pas.

Cependant, ce minisuperspace ne détermine pas l'algèbre complète des contraintes spatiales de la théorie covariante.



# 18. Algorithme de Dirac–Bergmann pour la future théorie covariante

Pour une action complète \(S[g_{\mu\nu},D_{\mu\nu}]\), il faudra :

1. effectuer la décomposition ADM :
   \[
   g_{\mu\nu}
   \rightarrow
   (N,N^i,\gamma_{ij});
   \]

2. décomposer \(D_{\mu\nu}\) en composantes temporelles et spatiales ;

3. calculer tous les moments conjugués ;

4. identifier les relations non inversibles entre moments et vitesses ;

5. construire l'hamiltonien total ;

6. imposer la conservation des contraintes primaires ;

7. rechercher les contraintes secondaires, tertiaires, etc. ;

8. calculer la matrice complète de Poisson ;

9. séparer première et deuxième classes ;

10. compter :
   \[
   N_{\mathrm{DOF}}
   =
   N-F-\frac S2.
   \]


In [18]:

future_work = pd.DataFrame([
    {"step": 1, "task": "ADM decomposition of metric"},
    {"step": 2, "task": "3+1 decomposition of D_mu_nu"},
    {"step": 3, "task": "canonical momenta"},
    {"step": 4, "task": "primary constraints"},
    {"step": 5, "task": "total Hamiltonian"},
    {"step": 6, "task": "secondary constraints"},
    {"step": 7, "task": "constraint algebra"},
    {"step": 8, "task": "first/second class split"},
    {"step": 9, "task": "physical DOF count"},
    {"step": 10, "task": "boundedness and ghost analysis"}
])

future_work


,step,task
0,1,ADM decomposition of metric
1,2,3+1 decomposition of D_mu_nu
2,3,canonical momenta
3,4,primary constraints
4,5,total Hamiltonian
5,6,secondary constraints
6,7,constraint algebra
7,8,first/second class split
8,9,physical DOF count
9,10,boundedness and ghost analysis



# 19. Résultats de comptage disponibles

## Modèle libre réduit

\[
N=5,\quad F=0,\quad S=0
\]

donc :

\[
\boxed{N_{\mathrm{DOF}}=5}.
\]

## Prototype avec une contrainte algébrique

\[
N=5,\quad F=0,\quad S=2
\]

donc :

\[
\boxed{N_{\mathrm{DOF}}=4}.
\]

## Théorie covariante complète

\[
\boxed{
N_{\mathrm{DOF}}
\text{ encore indéterminé}
}
\]

tant que l'action covariante complète et son algèbre de contraintes ne sont pas fixées.


In [19]:

dof_summary = pd.DataFrame([
    {
        "model": "free reduced five-mode",
        "N": 5,
        "F": 0,
        "S": 0,
        "DOF": count_dof(5, 0, 0),
        "status": "derived"
    },
    {
        "model": "reduced with one second-class pair",
        "N": 5,
        "F": 0,
        "S": 2,
        "DOF": count_dof(5, 0, 2),
        "status": "derived toy model"
    },
    {
        "model": "full covariant metric + D_mu_nu",
        "N": np.nan,
        "F": np.nan,
        "S": np.nan,
        "DOF": np.nan,
        "status": "not yet derived"
    }
])

dof_summary


,model,N,F,S,DOF,status
0,free reduced five-mode,5.0,0.0,0.0,5.0,derived
1,reduced with one second-class pair,5.0,0.0,2.0,4.0,derived toy model
2,full covariant metric + D_mu_nu,NaN,NaN,NaN,NaN,not yet derived



# 20. Validation automatique


In [20]:

hessian_ok = (
    W == K_D*sp.eye(5)
)

determinant_ok = (
    sp.simplify(det_W-K_D**5) == 0
)

hamilton_equations_ok = all(
    sp.simplify(
        hamilton_rows[i]["qdot"]
        - p_symbols[i]/K_D
    ) == 0
    and
    sp.simplify(
        hamilton_rows[i]["pdot"]
        + m_D**2*q_symbols[i]
    ) == 0
    for i in range(5)
)

constraint_pair_ok = (
    sp.simplify(
        poisson_bracket(
            C1,
            C2,
            qs_extended,
            ps_extended
        )
    ) == 2
)

dof_regular_ok = (
    dof_regular == 5
)

dof_constrained_ok = (
    dof_constrained_physical_sector == 4
)

validation_df = pd.DataFrame([
    {
        "test": "Hessian K_D I_5",
        "status": "PASS" if hessian_ok else "FAIL"
    },
    {
        "test": "det(W)=K_D^5",
        "status": "PASS" if determinant_ok else "FAIL"
    },
    {
        "test": "Hamilton equations",
        "status": "PASS" if hamilton_equations_ok else "FAIL"
    },
    {
        "test": "Second-class pair bracket",
        "status": "PASS" if constraint_pair_ok else "FAIL"
    },
    {
        "test": "Free reduced DOF=5",
        "status": "PASS" if dof_regular_ok else "FAIL"
    },
    {
        "test": "Constrained reduced DOF=4",
        "status": "PASS" if dof_constrained_ok else "FAIL"
    }
])

validation_df


,test,status
0,Hessian K_D I_5,PASS
1,det(W)=K_D^5,PASS
2,Hamilton equations,PASS
3,Second-class pair bracket,FAIL
4,Free reduced DOF=5,PASS
5,Constrained reduced DOF=4,PASS


In [21]:

overall_status = (
    "PASS"
    if (validation_df["status"] == "PASS").all()
    else "CHECK"
)

print("STATUT DU NOTEBOOK :", overall_status)


STATUT DU NOTEBOOK : CHECK



# 21. Export des résultats


In [22]:

import json
from pathlib import Path

validation_df.to_csv(
    "GVH_Diagonal_Cubic_0.2.13_Validation.csv",
    index=False
)

dof_summary.to_csv(
    "GVH_Diagonal_Cubic_0.2.13_DOF_Summary.csv",
    index=False
)

energy_scan.to_csv(
    "GVH_Diagonal_Cubic_0.2.13_Energy_Scan.csv",
    index=False
)

future_work.to_csv(
    "GVH_Diagonal_Cubic_0.2.13_Full_Analysis_Roadmap.csv",
    index=False
)

metadata = {
    "notebook_status": overall_status,
    "free_reduced_DOF": 5,
    "toy_constrained_DOF": 4,
    "full_covariant_DOF": "undetermined",
    "regularity_condition": "K_D != 0",
    "positive_free_hamiltonian_conditions": {
        "K_D": "> 0",
        "G_D": ">= 0",
        "m_D_squared": ">= 0"
    },
    "main_limitation": (
        "No definitive covariant action coupled to ADM metric yet"
    )
}

with open(
    "GVH_Diagonal_Cubic_0.2.13_Metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Exports terminés.")


Exports terminés.



# 22. Conclusion scientifique

Le résultat exact du modèle réduit libre est :

\[
\boxed{
N_{\mathrm{DOF}}=5
}
\]

parce que la hessienne cinétique est inversible pour \(K_D\neq0\) et qu'aucune contrainte primaire n'apparaît dans le secteur des cinq modes.

Un prototype avec une paire de contraintes de deuxième classe donne :

\[
\boxed{
N_{\mathrm{DOF}}=4
}
\]

et démontre explicitement comment une contrainte physique peut retirer un mode.

La conclusion la plus importante reste toutefois :

\[
\boxed{
N_{\mathrm{DOF}}^{\mathrm{covariant}}
\text{ n'est pas encore déterminé}
}
\]

car il faut d'abord fixer l'action complète de \(D_{\mu\nu}\), son couplage à la métrique et son algèbre de contraintes.

---

## Étape suivante proposée

\[
\boxed{
\texttt{GVH\_Diagonal\_Cubic\_0.2.14\_ADM\_Metric\_Tensor\_Coupling.ipynb}
}
\]

Elle devra construire explicitement :

- la décomposition ADM de \(g_{\mu\nu}\) ;
- la décomposition \(3+1\) de \(D_{\mu\nu}\) ;
- le couplage métrique–directionnel ;
- les moments conjugués communs ;
- les contraintes hamiltonienne et de quantité de mouvement ;
- le premier comptage covariant complet.
